In [ ]:
# ============================================================
# CELL 1 — Env guards (Colab / local)
# ============================================================
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
print("✅ Env guards set")

In [1]:
# ============================================================
# CELL 2 — Install dependencies (corpus construction only)
# ============================================================
!pip install -q datasets huggingface_hub cloudscraper beautifulsoup4
!pip install -q requests tqdm spacy
!pip install -q pandas numpy
!python -m spacy download it_core_news_sm -q

print("✅ Install done.")

✅ Stubbed google.protobuf.runtime_version


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

df_cicl path: /kaggle/input/datasets/mariabilik/df-cicl/df_cicl.csv
Device: cuda
  GPU: Tesla T4
  Mem: 15.6 GB
IFIT: 12 intents
✅ Tokenizer: CamembertTokenizerFast
   Vocab size: 32005
✅ Setup complete


In [2]:
# ============================================================
# CELL 3 — Imports, paths, seeds
# ============================================================
import os, sys, re, json, time, shutil, random, warnings, subprocess
from collections import Counter
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import spacy
from tqdm.auto import tqdm
import requests
import cloudscraper
from bs4 import BeautifulSoup
from huggingface_hub import hf_hub_download

# ---------- Paths ----------
OUT_DIR = os.environ.get('CIC_OUT_DIR', './')
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Seeds ----------
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# ---------- Gemini key ----------
GEMINI_KEY = None
try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_KEY = os.environ.get('GEMINI_API_KEY', None)
print(f"GEMINI_KEY available: {GEMINI_KEY is not None}")

# ---------- IFIT taxonomy ----------
ITALIAN_FRAUD_INTENTS = {
    'credential_request': 'Richiesta login/password/PIN/SPID',
    'payment_request':    'Richiesta pagamento/bonifico/dati carta/IBAN',
    'data_request':       'Richiesta codice fiscale/documento/indirizzo',
    'click_request':      'Richiesta click su link/allegato/QR code',
    'call_request':       'Richiesta chiamare un numero/contattare falso supporto',
    'urgency':            'Pressione temporale, scadenza',
    'authority':          'Riferimento a banca/Poste/INPS/polizia/ministero',
    'fear':               'Minaccia: blocco/multa/sanzione/azione legale',
    'greed':              'Promessa premio/bonus/rimborso/regalo',
    'impersonation':      'Fingere di essere conoscente/collega/parente',
    'social_proof':       'Riferimento ad altri utenti',
    'reciprocity':        'Senso di debito',
}
INTENT_CODES = list(ITALIAN_FRAUD_INTENTS.keys())
N_INTENTS = len(INTENT_CODES)

CHANNEL_MAP = {'email_fraud': 0, 'email_legit': 0,
               'sms_fraud': 1, 'forum': 3}

print(f"✅ Setup: {N_INTENTS} intents, seed={RANDOM_STATE}")

✅ SupCon sanity: loss=7.2350


In [3]:
# ============================================================
# CELL 4 — Forum corpus (Digital-Forum + MilanWorld)
# ============================================================
def build_forum_corpus(save_path=None):
    scraper = cloudscraper.create_scraper(
        browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False})

    def polite_get(url, delay=(2, 4)):
        time.sleep(random.uniform(*delay))
        try:
            r = scraper.get(url, timeout=45)
            return r if r.status_code == 200 else None
        except Exception:
            return None

    def parse_xenforo(soup):
        msgs = []
        for art in soup.find_all('article', class_='message'):
            content = art.find('div', class_='message-content')
            if not content:
                continue
            for q in content.find_all(['blockquote', 'aside', 'script', 'style']):
                q.decompose()
            text = re.sub(r'\s+', ' ',
                          content.get_text(strip=True, separator=' ')).strip()
            if 30 < len(text) < 5000:
                msgs.append(text)
        return msgs

    def scrape_thread(url, max_pages=25):
        all_msgs, base = [], url.rstrip('/')
        for page in range(1, max_pages + 1):
            page_url = base if page == 1 else f"{base}/page-{page}"
            r = polite_get(page_url)
            if not r:
                break
            soup = BeautifulSoup(r.content, 'html.parser')
            msgs = parse_xenforo(soup)
            if not msgs:
                break
            all_msgs.extend(msgs)
            print(f"  p.{page}: +{len(msgs)} (total {len(all_msgs)})")
            if not soup.find('a', class_='pageNav-jump--next'):
                break
        return all_msgs

    THREADS = [
        {"source": "digital_forum",
         "url": "https://www.digital-forum.it/threads/truffe-tramite-email-sms-app-qr-code-phishing-smishing-ecc.219519/"},
        {"source": "digital_forum",
         "url": "https://www.digital-forum.it/threads/attacco-hacker-a-gmail-tentativi-di-truffe-in-tutto-il-mondo-a-due-miliardi-di-caselle-di-posta.221024/"},
        {"source": "milanworld",
         "url": "https://www.milanworld.net/threads/italia-chiamate-spam-e-truffa-dalla-danimarca-45.158020/"},
    ]

    rows = []
    for i, t in enumerate(THREADS, 1):
        print(f"\n[{i}/{len(THREADS)}] {t['source']}")
        for m in scrape_thread(t["url"]):
            rows.append({"text": m, "source": t["source"], "url": t["url"]})

    df = pd.DataFrame(rows).drop_duplicates(subset='text').reset_index(drop=True)
    df = df[df['text'].str.len() > 100].reset_index(drop=True)
    df['text_length'] = df['text'].str.len()
    if save_path:
        df.to_csv(save_path, index=False)
    print(f"\nForum corpus: {len(df)} posts")
    return df


df_forum_raw = build_forum_corpus(
    save_path=os.path.join(OUT_DIR, 'forum_corpus_max.csv'))

✅ CICLMMDv2 ready


In [4]:
# ============================================================
# CELL 5 — Email, SMS, Telegram corpora
# ============================================================
# 5.1 E-PhishLLM (Italian email)
fp = hf_hub_download(repo_id="pajola/e-phishGen",
                     filename="ephishLLM.json", repo_type="dataset")
with open(fp) as f:
    data = json.load(f)
df_e = pd.DataFrame(data)
df_e = df_e[df_e['Language'] == 'it'].copy()
df_e['text'] = df_e['Subject'].fillna('') + ' ' + df_e['Body'].fillna('')
df_e = df_e[['text', 'type']].rename(columns={'type': 'target'})
df_e['source'] = 'email_ephish_it'
print(f"E-PhishLLM: {len(df_e)} | fraud={df_e['target'].sum()}")

# 5.2 Smishing-IMC25
if not os.path.exists('Smishing-Dataset-IMC25'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/reportsmishing/Smishing-Dataset-IMC25.git'],
                   check=False)
df_s = pd.read_csv('Smishing-Dataset-IMC25/dataset/final_dataset_output.csv')
df_s = df_s[df_s['language'].str.lower().str.contains('italian|^it$', na=False)].copy()
df_s = df_s[['text']].copy()
df_s['target'] = 1
df_s['source'] = 'sms_imc25_fraud'
print(f"Smishing: {len(df_s)} fraud SMS")

# 5.3 MuLTa-Telegram (legit, non-hateful)
if not os.path.exists('MuLTa-Telegram'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/dhfbk/MuLTa-Telegram.git'],
                   check=False)
with open('MuLTa-Telegram/data/it/train.json') as f:
    multa = json.load(f)
df_m = pd.DataFrame(multa)
df_m = df_m[df_m['hate label'] == 0][['text']].copy()
df_m['target'] = 0
df_m['source'] = 'multa_telegram'
print(f"MuLTa-Telegram: {len(df_m)} legit")

# 5.4 Forum
df_f = df_forum_raw.copy()
df_f['target'] = -1
df_f['source'] = df_f['source'] + '_forum'
print(f"Forum: {len(df_f)} posts")

# Save raw artefacts
for name, df in [('df_e_raw', df_e), ('df_s_raw', df_s),
                 ('df_m_raw', df_m), ('df_f_raw', df_f)]:
    df.to_csv(os.path.join(OUT_DIR, f'{name}.csv'), index=False)
print("✅ Raw corpora saved")

Loaded: 950 rows
Channels: {0: 421, 1: 416, 3: 113}
Mean intents/doc (all): 2.11
Mean intents/doc (n>0): 2.97

Forum subset (all):  n=113
Non-forum subset:    n=837
Forum intent support: 117 (mean=1.04)

[MAIN] Train: 665 | Val: 142 | Test: 143
[MAIN] Test channels: {1: 63, 0: 63, 3: 17}

[LOO-FORUM] Train: 711 | Val: 126 | Test (ALL forum): 113

[CV-FORUM] n_splits = 5 (strata sizes: {0: 68, 1: 45})
[CV-FORUM] Prepared 5 folds.
  fold 1: train=90 test=23
  fold 2: train=90 test=23
  fold 3: train=90 test=23
  fold 4: train=91 test=22
  fold 5: train=91 test=22


In [5]:
# ============================================================
# CELL 6 — Raw EDA sanity check
# ============================================================
print("="*70)
print("RAW DATA SANITY CHECK")
print("="*70)

summary = []
for name, df in [('email_ephish_it', df_e), ('sms_imc25_fraud', df_s),
                 ('multa_telegram', df_m), ('forum', df_f)]:
    row = {
        'source': name,
        'n': len(df),
        'n_missing_text': df['text'].isna().sum() if 'text' in df else 0,
        'n_dup_text': df['text'].duplicated().sum() if 'text' in df else 0,
        'len_mean': int(df['text'].dropna().str.len().mean()) if 'text' in df else 0,
        'len_median': int(df['text'].dropna().str.len().median()) if 'text' in df else 0,
    }
    if 'target' in df.columns:
        vc = df['target'].value_counts().to_dict()
        row['n_fraud'] = vc.get(1, 0)
        row['n_legit'] = vc.get(0, 0)
        row['n_unlabeled'] = vc.get(-1, 0)
    summary.append(row)

summary_df = pd.DataFrame(summary)
pd.set_option('display.max_columns', None, 'display.width', 200)
print(summary_df.to_string(index=False))

# Cross-source duplicates
all_texts = pd.concat([
    df_e[['text']].assign(src='email'),
    df_s[['text']].assign(src='sms'),
    df_m[['text']].assign(src='telegram'),
    df_f[['text']].assign(src='forum'),
], ignore_index=True)
dup = all_texts[all_texts.duplicated(subset='text', keep=False)]
if len(dup) > 0:
    print(f"\n⚠️  {dup['text'].nunique()} cross-source duplicates")
else:
    print("\n✅ No cross-source duplicates")

summary_df.to_csv(os.path.join(OUT_DIR, 'eda_raw_summary.csv'), index=False)

✅ Training utilities ready


In [7]:
# ============================================================
# CELL 7 — Preprocessing v7 (strict HTML, Cyrillic-aware)
# ============================================================
nlp = spacy.load("it_core_news_sm")
STOPWORDS_IT = list(nlp.Defaults.stop_words)

KNOWN_HTML = (
    r'html|head|body|div|span|p|a|b|i|u|br|hr|img|table|tr|td|th|ul|ol|li|'
    r'script|style|link|meta|form|input|button|textarea|select|option|'
    r'h[1-6]|nav|header|footer|section|article|strong|em|blockquote|pre|'
    r'code|iframe|video|audio|source|canvas|svg|main|aside|figure|figcaption|'
    r'mark|small|sub|sup|del|ins|time|address|dl|dt|dd|caption|thead|tbody|'
    r'tfoot|col|colgroup|fieldset|legend|label|optgroup|datalist|output'
)
HTML_RE = re.compile(rf'</?(?:{KNOWN_HTML})\b(?:\s+[^>]*=[^>]*)?>')
ENTITY_TAG_RE = re.compile(r'<([A-Za-z][A-Za-z0-9_]{1,49})>')
URL_RE   = re.compile(r'(?:https?://|www\.)\S+', re.IGNORECASE)
EMAIL_RE = re.compile(r'\S+@\S+\.\S+')
PHONE_RE = re.compile(r'(?:\+39[\s\-\.]?)?(?:3\d{2}[\s\-\.]?\d{3}[\s\-\.]?\d{3,4}'
                      r'|0\d{1,3}[\s\-\.]?\d{5,8})')
CYRILLIC_RE = re.compile(r'[\u0400-\u04FF\u0500-\u052F]')
WS_RE       = re.compile(r'\s+')
ZW_RE       = re.compile(r'[\u200b\u200c\u200d\ufeff\u2060]')

HOMOGLYPH_MAP = {
    'а':'a','е':'e','о':'o','р':'p','с':'c','у':'y','х':'x',
    'і':'i','ї':'i','є':'e','ґ':'g','ѕ':'s','ј':'j','ԁ':'d','ɡ':'g',
    'ⅰ':'i','ⅼ':'l','ⲟ':'o','ⲁ':'a',
}
HTML_ENTITIES = {
    '&amp;': ' & ', '&nbsp;': ' ', '&quot;': '"',
    '&lt;': ' < ', '&gt;': ' > ', '&#39;': "'", '&apos;': "'",
}
SPECIFIC_TAGS = {
    'url':      '[URL]',
    'email':    '[EMAIL]', 'email_address': '[EMAIL]',
    'phone':    '[PHONE]', 'phone_number':  '[PHONE]',
    'number':   '[NUMBER]', 'cardinal': '[NUMBER]',
    'ordinal':  '[NUMBER]', 'quantity': '[NUMBER]',
    'percent':  '[PERCENT]',
    'date':     '[DATE]', 'time': '[TIME]', 'date_time': '[DATE]',
    'money':    '[MONEY]',
}

def clean_text(text):
    if not isinstance(text, str):
        return "", {'was_none': True}
    flags = {k: False for k in ['had_html','had_url','had_email',
                                 'had_phone','had_cyrillic','had_homoglyph',
                                 'had_artifact','is_cyrillic_text',
                                 'is_mixed_text']}
    if HTML_RE.search(text):
        flags['had_html'] = True
        text = HTML_RE.sub(' ', text)

    def repl(m):
        tag = m.group(1).lower()
        return f" {SPECIFIC_TAGS.get(tag, '[ENTITY]')} "
    new = ENTITY_TAG_RE.sub(repl, text)
    if new != text:
        flags['had_artifact'] = True
        text = new

    for k, v in HTML_ENTITIES.items():
        if k in text:
            text = text.replace(k, v)
    if URL_RE.search(text):
        flags['had_url'] = True
        text = URL_RE.sub(' [URL] ', text)
    if EMAIL_RE.search(text):
        flags['had_email'] = True
        text = EMAIL_RE.sub(' [EMAIL] ', text)
    if PHONE_RE.search(text):
        flags['had_phone'] = True
        text = PHONE_RE.sub(' [PHONE] ', text)

    cyr_chars = CYRILLIC_RE.findall(text)
    if cyr_chars:
        flags['had_cyrillic'] = True
        n_cyr = len(cyr_chars)
        n_alpha = sum(c.isalpha() for c in text)
        pct_cyr = n_cyr / max(n_alpha, 1)
        if n_cyr >= 2 or pct_cyr >= 0.05:
            flags['is_cyrillic_text'] = True
        else:
            flags['had_homoglyph'] = True
            for k, v in HOMOGLYPH_MAP.items():
                text = text.replace(k, v)

    text = ZW_RE.sub('', text)
    text = WS_RE.sub(' ', text).strip().lower()
    return text, flags


def dedup_within(df, name):
    b = len(df)
    df = df.drop_duplicates(subset='text_clean', keep='first').reset_index(drop=True)
    if b - len(df):
        print(f"  {name:22s}: -{b-len(df)}")
    return df


def lemmatize_batch(texts, batch_size=256):
    docs = nlp.pipe(texts.astype(str), batch_size=batch_size,
                    disable=["ner", "parser"])
    return [" ".join(t.lemma_ for t in doc
                     if not t.is_stop and not t.is_punct
                     and not t.is_space and len(t.text) > 2)
            for doc in docs]


MARKERS = {
    'urgency':   ['immediatamente','urgente','entro','scadenza','subito',
                  'adesso','ultima','offerta','limitata','oggi','ore','tempo'],
    'authority': ['banca','poste','inps','agenzia','governo','ministero',
                  'polizia','tribunale','legale','avvocato','ufficiale',
                  'autorità','ente'],
    'fear':      ['bloccato','sospeso','perdita','multa','penale','problema',
                  'rischio','pericolo','allarme','violazione','cancellato',
                  'disattivato'],
    'greed':     ['premio','vincita','regalo','bonus','esclusiva','gratis',
                  'sconto','vantaggio','guadagno','investimento','rendimento'],
}

print("="*70)
print("PREPROCESSING v7")
print("="*70)

df_lab = pd.concat([df_e, df_s, df_m], ignore_index=True)
df_lab = df_lab.dropna(subset=['text']).reset_index(drop=True)
cleaned = df_lab['text'].apply(clean_text)
df_lab['text_clean'] = cleaned.apply(lambda x: x[0])
df_lab = pd.concat([df_lab, cleaned.apply(lambda x: x[1]).apply(pd.Series)], axis=1)

df_f = df_forum_raw.copy()
df_f['target'] = -1
df_f['source'] = df_f['source'] + '_forum'
df_f = df_f.dropna(subset=['text']).reset_index(drop=True)
cleaned_f = df_f['text'].apply(clean_text)
df_f['text_clean'] = cleaned_f.apply(lambda x: x[0])
df_f = pd.concat([df_f, cleaned_f.apply(lambda x: x[1]).apply(pd.Series)], axis=1)

n0_lab, n0_forum = len(df_lab), len(df_f)
print(f"Input: labeled={n0_lab} | forum={n0_forum}")

MIN_LEN = 30
df_lab = df_lab[df_lab['text_clean'].str.len() >= MIN_LEN].reset_index(drop=True)
df_f = df_f[df_f['text_clean'].str.len() >= MIN_LEN].reset_index(drop=True)

df_lab = (df_lab.groupby('source', group_keys=False)
          .apply(lambda g: dedup_within(g, g.name))
          .reset_index(drop=True))

prio = {'email_ephish_it': 0, 'sms_imc25_fraud': 1, 'multa_telegram': 2}
df_lab['_p'] = df_lab['source'].map(prio).fillna(9)
df_lab = (df_lab.sort_values(['text_clean', '_p'])
          .drop_duplicates('text_clean', keep='first')
          .drop(columns='_p').reset_index(drop=True))

df_f = df_f[~df_f['text_clean'].isin(set(df_lab['text_clean']))].reset_index(drop=True)

print("Lemmatizing...")
df_lab['text_lemma'] = lemmatize_batch(df_lab['text_clean'])
df_f['text_lemma']   = lemmatize_batch(df_f['text_clean'])
df_lab = df_lab[df_lab['text_lemma'].str.len() > 5].reset_index(drop=True)
df_f   = df_f[df_f['text_lemma'].str.len() > 5].reset_index(drop=True)

for df in [df_lab, df_f]:
    for name, words in MARKERS.items():
        pat = '|'.join(re.escape(w) for w in words)
        df[name] = df['text_clean'].str.count(pat)
    df['text_length'] = df['text_clean'].str.len()
    df['urgency_x_length'] = df['urgency'] * df['text_length']
    df['fear_x_length']    = df['fear']    * df['text_length']
    df['greed_x_length']   = df['greed']   * df['text_length']

print(f"\nFinal: labeled={len(df_lab)} | forum={len(df_f)}")
df_lab.to_csv(os.path.join(OUT_DIR, 'df_lab.csv'), index=False)
df_f.to_csv(os.path.join(OUT_DIR, 'df_f.csv'), index=False)
print(f"✅ Saved df_lab.csv ({len(df_lab)}), df_f.csv ({len(df_f)})")

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

✅ cicl_main resumed


In [ ]:
# ============================================================
# CELL 8 — IFIT annotation via Google Gemini
# ============================================================
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_API_KEY not set. Add Colab secret or env var.")

CANDIDATES = [
    'gemini-flash-latest',
    'gemini-2.5-flash',
    'gemini-2.5-flash-lite',
    'gemini-flash-lite-latest',
    'gemini-2.0-flash-001',
    'gemini-2.0-flash-lite',
]
API_VERSIONS = ['v1beta', 'v1']
WORKING = None

for api_ver in API_VERSIONS:
    for model_name in CANDIDATES:
        url = (f"https://generativelanguage.googleapis.com/{api_ver}/models/"
               f"{model_name}:generateContent?key={GEMINI_KEY}")
        payload = {"contents": [{"parts": [{"text": "Reply only with: OK"}]}],
                   "generationConfig": {"temperature": 0.0, "maxOutputTokens": 10}}
        try:
            r = requests.post(url, json=payload, timeout=15)
            if r.status_code == 200:
                print(f"  ✅ {api_ver}/{model_name}")
                WORKING = (model_name, api_ver)
                break
        except Exception:
            pass
    if WORKING:
        break

if not WORKING:
    raise RuntimeError("No working Gemini model found.")
MODEL_ANN, API_VER = WORKING
print(f"🎯 Using: {API_VER}/{MODEL_ANN}")

INTENT_BRIEF = "\n".join([f"- {k}: {v}" for k, v in ITALIAN_FRAUD_INTENTS.items()])
PROMPT = f"""Analizza il testo e identifica TUTTI gli intenti fraudolenti.

CODICI:
{INTENT_BRIEF}

REGOLE:
1. Multi-label (uno o più codici)
2. Rispondi SOLO con i codici separati da virgola
3. Se non ci sono intenti: NONE
4. Nessuna spiegazione

Testo:
\"\"\"
{{text}}
\"\"\"

Codici:"""

URL = (f"https://generativelanguage.googleapis.com/{API_VER}/models/"
       f"{MODEL_ANN}:generateContent?key={GEMINI_KEY}")
STATS = {'ok': 0, 'none': 0, 'rl': 0, 'err': 0, 'fail': 0}

def annotate_gemini(text, max_retries=3):
    if not isinstance(text, str) or len(text) < 30:
        return [], 'skip'
    payload = {
        "contents": [{"parts": [{"text": PROMPT.format(text=text[:2000])}]}],
        "generationConfig": {"temperature": 0.0, "maxOutputTokens": 200}
    }
    for attempt in range(max_retries):
        try:
            r = requests.post(URL, json=payload, timeout=30)
            if r.status_code == 429:
                STATS['rl'] += 1
                time.sleep(5 * (2 ** attempt))
                continue
            if r.status_code != 200:
                STATS['err'] += 1
                time.sleep(2)
                continue
            data = r.json()
            ans = data['candidates'][0]['content']['parts'][0]['text'].strip()
            if not ans:
                continue
            if re.search(r'\bNONE\b', ans, re.IGNORECASE):
                return [], 'NONE'
            pattern = '|'.join(re.escape(c) for c in INTENT_CODES)
            found = re.findall(pattern, ans.lower())
            seen, uniq = set(), []
            for c in found:
                if c not in seen:
                    seen.add(c); uniq.append(c)
            return uniq, ans
        except Exception:
            STATS['err'] += 1
            time.sleep(2)
    STATS['fail'] += 1
    return [], 'failed'

# Build corpus for annotation
df_lab_ann = df_lab.copy()
df_lab_ann['subset'] = df_lab_ann['source'].map({
    'email_ephish_it': None, 'sms_imc25_fraud': 'sms_fraud',
    'multa_telegram': None}).fillna(df_lab_ann['source'])
df_lab_ann.loc[(df_lab_ann['source'] == 'email_ephish_it') &
               (df_lab_ann['target'] == 1), 'subset'] = 'email_fraud'
df_lab_ann.loc[(df_lab_ann['source'] == 'email_ephish_it') &
               (df_lab_ann['target'] == 0), 'subset'] = 'email_legit'
df_lab_ann.loc[df_lab_ann['source'] == 'sms_imc25_fraud', 'subset'] = 'sms_fraud'

df_forum_ann = df_f.copy()
df_forum_ann['subset'] = 'forum'

df_lab_ann = df_lab_ann[df_lab_ann['subset'].isin(
    ['email_fraud', 'email_legit', 'sms_fraud'])].reset_index(drop=True)

df_ann = pd.concat([
    df_lab_ann[['text_clean', 'text_lemma', 'subset', 'source']],
    df_forum_ann[['text_clean', 'text_lemma', 'subset', 'source']],
], ignore_index=True)
df_ann['intent_labels'] = None
df_ann['raw_answer'] = ''

CKPT = os.path.join(OUT_DIR, 'annotation_checkpoint.csv')
SAVE_EVERY = 25
SLEEP = 4.5

print(f"\nAnnotating {len(df_ann)} documents...")
start = time.time()
for idx, row in tqdm(df_ann.iterrows(), total=len(df_ann), desc="Gemini"):
    if isinstance(row['intent_labels'], str) and row['intent_labels'].startswith('['):
        continue
    labels, raw = annotate_gemini(row['text_clean'])
    df_ann.at[idx, 'intent_labels'] = str(labels)
    df_ann.at[idx, 'raw_answer'] = raw
    if labels: STATS['ok'] += 1
    elif raw == 'NONE': STATS['none'] += 1
    if (idx + 1) % SAVE_EVERY == 0:
        df_ann.to_csv(CKPT, index=False)
    time.sleep(SLEEP)

df_ann.to_csv(os.path.join(OUT_DIR, 'annotation_full.csv'), index=False)
print(f"\n✅ Annotation complete in {(time.time()-start)/60:.1f} min")
print(f"   {STATS}")

In [ ]:
# ============================================================
# CELL 9 — Build final df_cicl.csv (multi-label target matrix)
# ============================================================
df = pd.read_csv(os.path.join(OUT_DIR, 'annotation_full.csv'))

def parse_labels(x):
    if not isinstance(x, str):
        return []
    try:
        v = eval(x)
        if isinstance(v, list):
            return [c for c in v if c in INTENT_CODES]
    except Exception:
        pass
    return []

df['labels_list'] = df['intent_labels'].apply(parse_labels)
df['n_intents'] = df['labels_list'].apply(len)

def to_multihot(labels):
    v = np.zeros(N_INTENTS, dtype=np.float32)
    for l in labels:
        v[INTENT_CODES.index(l)] = 1.0
    return v

df['multihot'] = df['labels_list'].apply(to_multihot)
df['channel_id'] = df['subset'].map(CHANNEL_MAP).fillna(0).astype(int)
df_cicl = df.copy().reset_index(drop=True)

print(f"Total: {len(df_cicl)}")
print(f"With intents: {(df_cicl['n_intents'] > 0).sum()}")
print(f"Mean intents: {df_cicl[df_cicl['n_intents']>0]['n_intents'].mean():.2f}")

df_cicl.to_csv(os.path.join(OUT_DIR, 'df_cicl.csv'), index=False)
print(f"✅ Saved df_cicl.csv ({len(df_cicl)} rows)")
print("📥 Download df_cicl.csv and upload as Kaggle dataset mariabilik/df-cicl")